In [ ]:

from torchvision import transforms

from __future__ import print_function




In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
import matplotlib.pyplot as plt



In [ ]:

import torchvision.transforms as transforms

import torchvision.models as models


In [ ]:


import os
import torch
import imageio
import torchvision

from torch.utils.data import Dataset

from torchvision import transforms

from sklearn.model_selection import train_test_split

from PIL import Image

import torchvision.transforms as T

import matplotlib.pyplot as plt
import numpy as np
## from torcheval.metrics.functional import multiclass_f1_score as f1_score




In [ ]:

raw_data_train = '/content/drive/MyDrive/FruitsAdversarialML'

raw_data_test  = '/content/drive/MyDrive/FruitsAdversarialML'



In [ ]:

PATH = "/content/drive/MyDrive/"

lr = 0.00001


In [ ]:


dataset_train = []
labels_train  = []
targets_train = []

for folder in sorted( os.listdir( raw_data_train ) ):
    ## print(folder)
    for image in sorted(os.listdir( os.path.join(raw_data_train, folder) )):
        if folder not in labels_train:
            labels_train.append( folder )
        targets_train.append(  labels_train.index(folder)  )
        img_arr = imageio.imread(  os.path.join(raw_data_train, folder, image), pilmode="RGB"  )

        img = torch.from_numpy( img_arr ).permute( 2, 0, 1 ).float()

        img /= 255
        dataset_train.append(img)



In [ ]:

data_train    = torch.stack( dataset_train )
targets_train = torch.Tensor(  targets_train  ).type(   torch.LongTensor   )

torch.save(   (data_train, targets_train, labels_train), "InClass_CIFAR10_data"     )

## data1, targets1, labels1 = torch.load("InClass_CIFAR10_data")



In [ ]:

X_train = data_train
y_train = targets_train




In [ ]:


X_train = X_train.numpy()
##X_test  = X_test.numpy()

X_train = X_train.astype(  np.float32  )
##X_test  = X_test.astype(   np.float32  )

X_train = torch.from_numpy(X_train )
##X_test = torch.from_numpy( X_test  )


In [ ]:


X_train.shape



In [ ]:

preprocess = transforms.Compose([
                 ## transforms.Resize(256),
                 transforms.CenterCrop(224),
                 # transforms.ToTensor()
])


X_train = preprocess(X_train)




In [ ]:

CIFAR_train_list = [  ( X_train[i],  y_train[i].item() )  for i in range( X_train.shape[0]   )  ]



In [ ]:

batch_size = 64  ## 16


In [ ]:

train_dl = torch.utils.data.DataLoader( CIFAR_train_list, batch_size=batch_size, shuffle=True  )



In [ ]:

model = models.vit_b_32( weights=models.ViT_B_32_Weights.DEFAULT)



In [ ]:

model.heads = torch.nn.Linear(
    in_features=model.heads.head.in_features,
    out_features=10

)



In [ ]:

torch.nn.init.xavier_uniform_( model.heads.weight )



In [ ]:


print( model )


In [ ]:

model.train()


In [ ]:

loss_fn = nn.CrossEntropyLoss()


In [ ]:

params = [ param for name, param in model.named_parameters() if 'heads' not in str(name)]


In [ ]:


opt = torch.optim.Adam(  [{'params': params}, {'params': model.heads.parameters(), 'lr': lr*10}], lr=lr) ## , weight_decay=weight_decay   )





In [ ]:

def training_loop( N_Epochs, model, loss_fn, opt ):
    for epoch in range(N_Epochs):
        for xb, yb in train_dl:

            ## xb = xb.view(  (16, -1 ) )

            ## xb = xb  ##.to( torch_device )
            ## yb = yb  ## .to( torch_device )

            xb = xb.to(torch_device)
            yb = yb.to(torch_device)

            y_pred = model(xb)

            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % 1 == 0:
            print(epoch, "loss=", loss)
            new_PATH = PATH + str(epoch)
            print( new_PATH )
            torch.save(model, new_PATH)



In [ ]:

N_Epochs = 10
torch_device = 'cuda'

model.to(torch_device)

params = [param for name, param in model.named_parameters() if 'heads' not in str(name)]

opt = torch.optim.Adam(
    [{'params': params}, {'params': model.heads.parameters(), 'lr': lr*10}],
    lr=lr
)



training_loop( N_Epochs, model, loss_fn, opt )



In [ ]:

model.eval()

def predict_image(img_path):
    img = imageio.imread(img_path)
    
    transform = T.Compose([
        T.ToPILImage(),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    
    img = transform(img).unsqueeze(0).to(torch_device)
    
    with torch.no_grad():
        out = model(img)
        pred = torch.argmax(out, dim=1).item()
    
    return pred

# example
pred = predict_image("/content/drive/MyDrive/FruitsAdversarialML/banana/100_100.jpg")
print("Predicted class:", pred)



In [ ]:

from sklearn.metrics import f1_score, confusion_matrix

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in train_dl:   # use test_dl if you have one
        xb = xb.to(torch_device)
        yb = yb.to(torch_device)

        out = model(xb)
        preds = torch.argmax(out, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

# F1
f1 = f1_score(all_targets, all_preds, average='macro')
print("F1 score:", f1)

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
print("Confusion Matrix:\n", cm)



In [ ]:

import matplotlib.pyplot as plt

plt.imshow(cm)
plt.colorbar()
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
